In [ ]:
from pathlib import Path
import subprocess
import sys
project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(project_root / 'requirements.txt')])


In [ ]:
from pathlib import Path
import os
import random
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
MODELS_DIR = PROJECT_ROOT / 'models'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'
print(f'Seed fixed at {SEED}')
print(f'Project root: {PROJECT_ROOT}')


# Notebook 01 ? Data Download + Multi-Hazard Catalog
## Mapathon 2026 | CASA-Net Bangladesh Multi-Hazard Pipeline


## Section 1.1 — Environment Setup

This matters because every downstream notebook expects the same directory structure, environment variables, and saved intermediates. Establishing those assumptions here makes the rest of the stack top-to-bottom runnable.


In [ ]:
import json
import zipfile
from datetime import datetime

import geopandas as gpd
import pandas as pd
import requests
from dotenv import load_dotenv

CONFIG_DIR = PROJECT_ROOT / 'config'
CATALOG_PATH = CONFIG_DIR / 'study_areas_years.json'
RAW_SENTINEL = RAW_DIR / 'sentinel1'
RAW_UNOSAT = RAW_DIR / 'unosat'
RAW_CEMS = RAW_DIR / 'cems'
RAW_FFWC = RAW_DIR / 'ffwc'
RAW_INFRA = RAW_DIR / 'infrastructure'
RAW_POP = RAW_DIR / 'population'
RAW_DEM = RAW_DIR / 'dem'
PROCESSED_SAR = PROCESSED_DIR / 'sar'
PROCESSED_TERRAIN = PROCESSED_DIR / 'terrain'
PROCESSED_MASKS = PROCESSED_DIR / 'masks'
PROCESSED_PATCHES = PROCESSED_DIR / 'patches'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
MAPS_DIR = OUTPUTS_DIR / 'maps'
REPORT_DIR = OUTPUTS_DIR / 'report'
for folder in [CONFIG_DIR, RAW_SENTINEL, RAW_UNOSAT, RAW_CEMS, RAW_FFWC, RAW_INFRA, RAW_POP, RAW_DEM, PROCESSED_SAR, PROCESSED_TERRAIN, PROCESSED_MASKS, PROCESSED_PATCHES, FIGURES_DIR, MAPS_DIR, REPORT_DIR, MODELS_DIR, NOTEBOOKS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

load_dotenv(PROJECT_ROOT / '.env')
CATALOG = json.loads(CATALOG_PATH.read_text())
YEARS = CATALOG['years']
REGIONS = CATALOG['regions']
WINDOW_TEMPLATES = CATALOG['window_templates']
REGISTRY_PATH = REPORT_DIR / 'download_registry.json'
registry = json.loads(REGISTRY_PATH.read_text()) if REGISTRY_PATH.exists() else {}

def note_status(name, status, target=None, detail=None, manual_url=None):
    registry[name] = {'status': status, 'target': str(target) if target else None, 'detail': detail, 'manual_url': manual_url, 'updated_at': datetime.utcnow().isoformat(timespec='seconds') + 'Z'}

def stream_download(url, target_path):
    response = requests.get(url, timeout=300, stream=True)
    response.raise_for_status()
    with open(target_path, 'wb') as fh:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                fh.write(chunk)
    return target_path

def build_scene_requests():
    rows = []
    for region in REGIONS:
        for year in YEARS:
            for window_name, (start_mmdd, end_mmdd) in WINDOW_TEMPLATES.items():
                rows.append({'region_id': region['id'], 'region_name': region['name'], 'year': year, 'window': window_name, 'start_date': f'{year}-{start_mmdd}', 'end_date': f'{year}-{end_mmdd}', 'bbox': region['bbox'], 'stations': ','.join(region.get('stations', []))})
    return pd.DataFrame(rows)

scene_requests_df = build_scene_requests()
scene_requests_df.to_csv(REPORT_DIR / 'scene_request_manifest.csv', index=False)
label_manifest_df = pd.DataFrame([{'label_source': source, 'manual_location': 'Place shapefiles or rasters under data/raw/unosat or data/raw/cems'} for source in CATALOG.get('label_sources', [])])
label_manifest_df.to_csv(REPORT_DIR / 'label_source_manifest.csv', index=False)
print(f'Regions: {len(REGIONS)} | Years: {YEARS[0]}-{YEARS[-1]} | Scene requests: {len(scene_requests_df)}')
display(scene_requests_df.head())


## Section 1.2 — Sentinel-1 Download

Sentinel-1 GRD VV/VH is the core imagery source for CASA-Net. This section searches Copernicus Data Space, saves a scene manifest, and attempts authenticated download when `.env` credentials are available.


In [ ]:
CDSE_CATALOG = 'https://catalogue.dataspace.copernicus.eu/odata/v1/Products'
CDSE_TOKEN_URL = os.getenv('CDSE_TOKEN_URL', 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token')
CDSE_CLIENT_ID = os.getenv('CDSE_CLIENT_ID', 'cdse-public')
CDSE_USERNAME = os.getenv('CDSE_USERNAME')
CDSE_PASSWORD = os.getenv('CDSE_PASSWORD')

def bbox_to_polygon(bbox):
    west, south, east, north = bbox
    return f"POLYGON(({west} {south}, {east} {south}, {east} {north}, {west} {north}, {west} {south}))"

def build_filter(row):
    return (
        "Collection/Name eq 'SENTINEL-1' and "
        "Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' and att/OData.CSC.StringAttribute/Value eq 'GRD') and "
        "Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'polarisationChannels' and att/OData.CSC.StringAttribute/Value eq 'VV&VH') and "
        "Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'sensorMode' and att/OData.CSC.StringAttribute/Value eq 'IW') and "
        f"OData.CSC.Intersects(area=geography'SRID=4326;{bbox_to_polygon(row['bbox'])}') and "
        f"ContentDate/Start ge {row['start_date']}T00:00:00.000Z and ContentDate/Start le {row['end_date']}T23:59:59.999Z"
    )

def get_token():
    if not CDSE_USERNAME or not CDSE_PASSWORD:
        raise RuntimeError('Missing CDSE_USERNAME/CDSE_PASSWORD in .env')
    response = requests.post(CDSE_TOKEN_URL, data={'grant_type': 'password', 'client_id': CDSE_CLIENT_ID, 'username': CDSE_USERNAME, 'password': CDSE_PASSWORD}, timeout=60)
    response.raise_for_status()
    return response.json()['access_token']

try:
    rows = []
    for _, row in scene_requests_df.iterrows():
        query_row = row.to_dict(); query_row['bbox'] = row['bbox'] if isinstance(row['bbox'], list) else eval(row['bbox'])
        response = requests.get(CDSE_CATALOG, params={'$filter': build_filter(query_row), '$top': 10, '$orderby': 'ContentDate/Start asc'}, timeout=60)
        response.raise_for_status()
        for item in response.json().get('value', []):
            rows.append({'region_id': row['region_id'], 'region_name': row['region_name'], 'year': row['year'], 'window': row['window'], 'name': item.get('Name'), 'product_id': item.get('Id'), 'sensing_start': item.get('ContentDate', {}).get('Start')})
    sentinel_df = pd.DataFrame(rows).drop_duplicates(subset=['name'])
    sentinel_df.to_csv(RAW_SENTINEL / 'sentinel1_scene_manifest.csv', index=False)
    display(sentinel_df.head(20))
    note_status('sentinel1_manifest', 'success', RAW_SENTINEL / 'sentinel1_scene_manifest.csv', f'{len(sentinel_df)} scenes found across {len(REGIONS)} regions and {len(YEARS)} years', 'https://dataspace.copernicus.eu/browser/')
except Exception as exc:
    sentinel_df = pd.DataFrame()
    print('Sentinel-1 search failed. Manual fallback: https://dataspace.copernicus.eu/browser/')
    print(exc)
    note_status('sentinel1_manifest', 'failed', RAW_SENTINEL, str(exc), 'https://dataspace.copernicus.eu/browser/')

try:
    if not sentinel_df.empty:
        token = get_token(); headers = {'Authorization': f'Bearer {token}'}
        for _, row in sentinel_df.iterrows():
            target = RAW_SENTINEL / f"{row['name']}.zip"
            if target.exists():
                continue
            response = requests.get(f"https://download.dataspace.copernicus.eu/odata/v1/Products({row['product_id']})/$value", headers=headers, timeout=600, stream=True)
            response.raise_for_status()
            with open(target, 'wb') as fh:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        fh.write(chunk)
        note_status('sentinel1_download', 'success', RAW_SENTINEL, 'Downloaded or confirmed all listed Sentinel-1 archives.', 'https://dataspace.copernicus.eu/browser/')
except Exception as exc:
    print('Sentinel-1 authenticated download skipped or failed. Manual fallback: https://dataspace.copernicus.eu/browser/')
    print(exc)
    note_status('sentinel1_download', 'failed', RAW_SENTINEL, str(exc), 'https://dataspace.copernicus.eu/browser/')


## Section 1.3 — UNOSAT Flood Mask Download

The UNOSAT Sylhet flood extent is the main supervision source for June 2024. We save the original files and inspect the shapefile immediately so later notebooks inherit a validated label source.


In [ ]:
UNOSAT_PAGE = 'https://data.humdata.org/dataset/cumulative-satellite-detected-waters-over-sylhet-division-bangladesh-as-of-19-22-june-2024'
UNOSAT_DIRECT = os.getenv('UNOSAT_DIRECT_URL')
try:
    if UNOSAT_DIRECT and not (RAW_UNOSAT / 'unosat_sylhet_june_2024.zip').exists():
        stream_download(UNOSAT_DIRECT, RAW_UNOSAT / 'unosat_sylhet_june_2024.zip')
        if zipfile.is_zipfile(RAW_UNOSAT / 'unosat_sylhet_june_2024.zip'):
            with zipfile.ZipFile(RAW_UNOSAT / 'unosat_sylhet_june_2024.zip') as zf:
                zf.extractall(RAW_UNOSAT)
    shapefiles = list(RAW_UNOSAT.rglob('*.shp'))
    if not shapefiles:
        raise FileNotFoundError('No shapefile found. Set UNOSAT_DIRECT_URL or download manually.')
    unosat_gdf = gpd.read_file(shapefiles[0])
    total_area_km2 = unosat_gdf.to_crs(32646).geometry.area.sum() / 1e6
    print(f'Polygon count: {len(unosat_gdf):,}')
    print(f'Total area km²: {total_area_km2:,.2f}')
    print(f'CRS: {unosat_gdf.crs}')
    note_status('unosat', 'success', shapefiles[0], f'{len(unosat_gdf)} polygons | {total_area_km2:.2f} km² | CRS={unosat_gdf.crs}', UNOSAT_PAGE)
except Exception as exc:
    print(f'UNOSAT fallback: {UNOSAT_PAGE}')
    print(exc)
    note_status('unosat', 'failed', RAW_UNOSAT, str(exc), UNOSAT_PAGE)


## Section 1.4 — FFWC River Gauge Download

Gauge data is needed later for threshold calibration and anticipatory-action lead time analysis. Saving it now lets the threshold notebook work from local CSVs only.


In [ ]:
FFWC_API = os.getenv('FFWC_API_URL', 'https://ffwc.bwdb.gov.bd/data_load/')
STATIONS = {
    'Kanaighat': {'danger_m': 12.75, 'code': os.getenv('FFWC_KANAIGHAT_CODE', 'Kanaighat')},
    'Sylhet': {'danger_m': 10.80, 'code': os.getenv('FFWC_SYLHET_CODE', 'Sylhet')},
    'Sunamganj': {'danger_m': 7.80, 'code': os.getenv('FFWC_SUNAMGANJ_CODE', 'Sunamganj')},
    'Sheola': {'danger_m': 13.05, 'code': os.getenv('FFWC_SHEOLA_CODE', 'Sheola')},
}
try:
    frames = []
    for station, meta in STATIONS.items():
        response = requests.get(FFWC_API, params={'station': meta['code'], 'from': '2024-05-01', 'to': '2024-09-30', 'format': 'json'}, timeout=120)
        response.raise_for_status()
        payload = response.json()
        rows = payload.get('data', payload if isinstance(payload, list) else [])
        df = pd.DataFrame(rows)
        if df.empty:
            raise ValueError(f'No rows returned for {station}')
        df['station'] = station
        df['danger_m'] = meta['danger_m']
        frames.append(df)
    ffwc_df = pd.concat(frames, ignore_index=True)
    ffwc_df.to_csv(RAW_FFWC / 'sylhet_stations_2024.csv', index=False)
    display(ffwc_df.head())
    note_status('ffwc', 'success', RAW_FFWC / 'sylhet_stations_2024.csv', f'{len(ffwc_df)} rows across {len(STATIONS)} stations', 'https://ffwc.bwdb.gov.bd/data_load/')
except Exception as exc:
    print('FFWC fallback: https://ffwc.bwdb.gov.bd/data_load/ and http://hydrology.bwdb.gov.bd/')
    print(exc)
    note_status('ffwc', 'failed', RAW_FFWC / 'sylhet_stations_2024.csv', str(exc), 'https://ffwc.bwdb.gov.bd/data_load/')


## Section 1.5 — Infrastructure Download

Infrastructure layers support the child-centred impact analysis. The notebook accepts direct file URLs from `.env` because HDX resource links often change while the dataset landing pages stay stable.


In [ ]:
infra_specs = {
    'health.geojson': ('HDX_HEALTH_URL', 'https://data.humdata.org/group/bgd'),
    'education.geojson': ('HDX_EDUCATION_URL', 'https://data.humdata.org/group/bgd'),
    'shelters.geojson': ('HDX_SHELTERS_URL', 'https://data.humdata.org/group/bgd'),
    'admin_union.zip': ('HDX_ADMIN_UNION_URL', 'https://data.humdata.org/group/bgd'),
}
infra_rows = []
for filename, (env_key, manual_url) in infra_specs.items():
    target = RAW_INFRA / filename
    try:
        direct_url = os.getenv(env_key)
        if direct_url and not target.exists():
            stream_download(direct_url, target)
            if target.suffix == '.zip' and zipfile.is_zipfile(target):
                with zipfile.ZipFile(target) as zf:
                    zf.extractall(RAW_INFRA)
        if filename.endswith('.geojson'):
            path = RAW_INFRA / filename
            if not path.exists():
                raise FileNotFoundError(f'Missing {filename}; set {env_key} or download manually.')
            gdf = gpd.read_file(path)
        else:
            shp_candidates = list(RAW_INFRA.rglob('*.shp'))
            if not shp_candidates:
                raise FileNotFoundError('Admin union shapefile not found after download.')
            gdf = gpd.read_file(shp_candidates[0])
        infra_rows.append({'layer': filename, 'count': len(gdf), 'crs': str(gdf.crs)})
        note_status(f'infra_{filename}', 'success', target, f'{len(gdf)} features | CRS={gdf.crs}', manual_url)
    except Exception as exc:
        print(f'{filename} fallback: {manual_url}')
        print(exc)
        note_status(f'infra_{filename}', 'failed', target, str(exc), manual_url)
if infra_rows:
    display(pd.DataFrame(infra_rows))


## Section 1.6 — WorldPop Download

The population raster lets us estimate total and child exposure at union level later in the workflow. Keeping the download step here means the impact notebook can stay analytical instead of API-heavy.


In [ ]:
WORLDPOP_PAGE = 'https://hub.worldpop.org/geodata/summary?id=94'
WORLDPOP_DIRECT = os.getenv('WORLDPOP_TOTAL_URL')
try:
    target = RAW_POP / 'bgd_ppp_2020_100m.tif'
    if WORLDPOP_DIRECT and not target.exists():
        stream_download(WORLDPOP_DIRECT, target)
    if not target.exists():
        raise FileNotFoundError('WorldPop raster not found. Set WORLDPOP_TOTAL_URL or download manually.')
    print(target)
    note_status('worldpop', 'success', target, 'Population raster ready for overlay analysis.', WORLDPOP_PAGE)
except Exception as exc:
    print(f'WorldPop fallback: {WORLDPOP_PAGE}')
    print(exc)
    note_status('worldpop', 'failed', RAW_POP / 'bgd_ppp_2020_100m.tif', str(exc), WORLDPOP_PAGE)


## Section 1.7 — DEM and Terrain Layer Download

SRTM and JRC permanent water provide the physical context needed by the Haor Terrain Conditioning branch. This notebook only downloads the raw sources; the derived slope, TWI, and HAND rasters are created in Notebook 02.


In [ ]:
OT_URL = 'https://portal.opentopography.org/API/globaldem'
OT_KEY = os.getenv('OPENTOPOGRAPHY_API_KEY')
JRC_PAGE = 'https://global-surface-water.appspot.com/download'
JRC_DIRECT = os.getenv('JRC_PERMANENT_WATER_URL')
try:
    dem_target = RAW_DEM / 'srtm_sylhet_30m.tif'
    if OT_KEY and not dem_target.exists():
        response = requests.get(OT_URL, params={'demtype':'SRTMGL1', 'south':AOI_BBOX[1], 'north':AOI_BBOX[3], 'west':AOI_BBOX[0], 'east':AOI_BBOX[2], 'outputFormat':'GTiff', 'API_Key':OT_KEY}, timeout=600, stream=True)
        response.raise_for_status()
        with open(dem_target, 'wb') as fh:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    fh.write(chunk)
    if not dem_target.exists():
        raise FileNotFoundError('DEM missing. Set OPENTOPOGRAPHY_API_KEY or download manually from EarthExplorer/OpenTopography.')
    note_status('dem', 'success', dem_target, 'SRTM DEM saved for terrain derivation.', 'https://earthexplorer.usgs.gov/')
except Exception as exc:
    print('DEM fallbacks: https://opentopography.org/ and https://earthexplorer.usgs.gov/')
    print(exc)
    note_status('dem', 'failed', RAW_DEM / 'srtm_sylhet_30m.tif', str(exc), 'https://earthexplorer.usgs.gov/')

try:
    jrc_target = RAW_DEM / 'jrc_permanent_water.tif'
    if JRC_DIRECT and not jrc_target.exists():
        stream_download(JRC_DIRECT, jrc_target)
    if not jrc_target.exists():
        raise FileNotFoundError('JRC raster missing. Set JRC_PERMANENT_WATER_URL or download manually.')
    note_status('jrc_permanent_water', 'success', jrc_target, 'JRC permanent water ready for Notebook 02.', JRC_PAGE)
except Exception as exc:
    print(f'JRC fallback: {JRC_PAGE}')
    print(exc)
    note_status('jrc_permanent_water', 'failed', RAW_DEM / 'jrc_permanent_water.tif', str(exc), JRC_PAGE)


## Section 1.8 — Download Summary Checklist

A saved status table makes it easy to see what still needs manual intervention before preprocessing starts. That is especially useful when the notebooks are handed off or rerun on another machine.


In [ ]:
REGISTRY_PATH.write_text(json.dumps(registry, indent=2))
summary_df = pd.DataFrame.from_dict(registry, orient='index').reset_index(names='dataset')
summary_df['status_icon'] = summary_df['status'].map({'success':'✅','failed':'❌'}).fillna('⚪')
summary_df.to_csv(REPORT_DIR / 'download_summary.csv', index=False)
display(summary_df[['status_icon','dataset','status','target','detail','manual_url']].sort_values(['status','dataset']))
print(f"Saved summary: {REPORT_DIR / 'download_summary.csv'}")


<!-- MULTI-HAZARD EXTENSION GENERATED -->
## Section 1.9 ? Multi-Hazard Dataset Catalog
This section upgrades the project from flood-only downloads to a **multi-hazard data registry**. Flood remains the validated branch, while erosion and landslide data sources are now tracked in the same project tree so the notebooks can scale into a shared hazard workflow.


In [ ]:
# MULTI-HAZARD EXTENSION GENERATED
import os
import pandas as pd
from analysis.multi_hazard_support import load_hazard_catalog, ensure_multihazard_tree, ordered_hazard_ids

ROOT = Path(r'f:\MAPATHON\sylhet_flood_2024')
hazard_catalog = load_hazard_catalog(ROOT / 'config' / 'hazard_catalog.json')
ensure_multihazard_tree(ROOT, hazard_catalog)

dataset_rows = []
for hazard in hazard_catalog['hazards']:
    for url in hazard['source_urls']:
        dataset_rows.append({
            'hazard': hazard['id'],
            'hazard_name': hazard['name'],
            'task': hazard['task'],
            'source_url': url,
            'env_vars': ', '.join(hazard['env_vars']),
            'inputs': ', '.join(hazard['inputs']),
            'outputs': ', '.join(hazard['outputs'])
        })
multi_hazard_dataset_manifest = pd.DataFrame(dataset_rows)
multi_hazard_dataset_manifest.to_csv(ROOT / 'outputs' / 'report' / 'multi_hazard_dataset_manifest.csv', index=False)
multi_hazard_dataset_manifest


<!-- MULTI-HAZARD EXTENSION GENERATED -->
## Section 1.10 ? Multi-Hazard Download Readiness
Each hazard has different external data needs. This checklist lets the notebook run safely even before every hazard branch is populated with labels.


In [ ]:
# MULTI-HAZARD EXTENSION GENERATED
checklist_rows = []
for hazard in hazard_catalog['hazards']:
    for env_key in hazard['env_vars']:
        checklist_rows.append({
            'hazard': hazard['id'],
            'env_var': env_key,
            'configured': bool(os.getenv(env_key)),
        })
multi_hazard_download_checklist = pd.DataFrame(checklist_rows)
multi_hazard_download_checklist.to_csv(ROOT / 'outputs' / 'report' / 'multi_hazard_download_checklist.csv', index=False)
multi_hazard_download_checklist
